> **Scratch notebook.** The analysis here led to `bandaid.ripple` (scan-time harmonic model). The `destripe_*` functions used at the end are deprecated wrappers around `destripe_profile`. Paths are relative to the repo root; run from `notebooks/` after `pip install -e .`.

# Stripe pattern analysis
Goal: figure out whether the vertical banding is a clean sinusoid (one frequency) or a multi-harmonic pattern (square-wave-like). This determines whether we need a single notch filter or a comb filter.\n\nThen walk through the per-frame Fourier notch destriping approach step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

from bandaid.io import load_tiff_stack

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Load data
Change the path to point to a file with visible banding.

In [ ]:
tiff_path = Path("./data/HK13/HK13_20260814/HK13_20260814_50pw_00002.tif")
# tiff_path = Path("/Users/hakan/Downloads/AMD10/AMD10_20260618/AMD10_20260618_002/AMD10_20260618_00002.tif")
mov = load_tiff_stack(tiff_path)
print(f"Shape: {mov.shape}, dtype: {mov.dtype}")

## 1. Visualize the stripes
Look at a single frame and the temporal mean to see the banding.

In [ ]:
frame_idx = 0
frame = mov[frame_idx].astype(np.float32)
tmean = mov.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(frame, cmap='gray', vmin=np.percentile(frame, 1), vmax=np.percentile(frame, 99))
axes[0].set_title(f'Single frame ({frame_idx})')
axes[1].imshow(tmean, cmap='gray', vmin=np.percentile(tmean, 1), vmax=np.percentile(tmean, 99))
axes[1].set_title('Temporal mean')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

## 2. Extract column profiles
Average each column across all rows → 1D stripe profile. Compare a few frames to see drift.

In [ ]:
# Column profiles for a few frames spread across the recording
n_show = 5
# frame_indices = np.linspace(0, mov.shape[0] - 1, n_show, dtype=int)
frame_indices = np.arange(1,5)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [2, 1]})

# Raw column profiles
ax = axes[0]
for idx in frame_indices:
    profile = mov[idx].astype(np.float32).mean(axis=0)
    ax.plot(profile, alpha=0.7, label=f'frame {idx}')
ax.set_xlabel('Column')
ax.set_ylabel('Mean intensity')
ax.set_title('Column profiles (mean across rows)')
ax.legend(fontsize=8)

# De-meaned to see just the stripe pattern
ax = axes[1]
for idx in frame_indices:
    profile = mov[idx].astype(np.float32).mean(axis=0)
    profile -= profile.mean()
    ax.plot(profile, alpha=0.7, label=f'frame {idx}')
ax.set_xlabel('Column')
ax.set_ylabel('Deviation from mean')
ax.set_title('De-meaned column profiles (stripe pattern only)')
ax.legend(fontsize=8)

plt.tight_layout()

## 3. Frequency content — sinusoid or multi-harmonic?
FFT of the column profile. A pure sinusoid = one peak. Square wave or sawtooth = fundamental + harmonics at 2f, 3f, etc.

In [ ]:
# FFT of column profiles from several frames
fig, axes = plt.subplots(len(frame_indices), 2, figsize=(12, 3 * len(frame_indices)))

for row, idx in enumerate(frame_indices):
    profile = mov[idx].astype(np.float32).mean(axis=0)
    profile -= profile.mean()
    
    W = len(profile)
    freqs = np.fft.rfftfreq(W)  # in cycles/pixel
    magnitude = np.abs(np.fft.rfft(profile))
    magnitude[0] = 0  # drop DC
    
    # Spatial domain
    axes[row, 0].plot(profile)
    axes[row, 0].set_title(f'Frame {idx} — column profile')
    axes[row, 0].set_xlabel('Column')
    
    # Frequency domain
    axes[row, 1].plot(freqs, magnitude)
    axes[row, 1].set_title(f'Frame {idx} — FFT magnitude')
    axes[row, 1].set_xlabel('Frequency (cycles/pixel)')
    axes[row, 1].set_ylabel('Magnitude')
    
    # Mark peaks
    peak_threshold = magnitude.max() * 0.2
    peak_mask = magnitude > peak_threshold
    peak_freqs = freqs[peak_mask]
    peak_mags = magnitude[peak_mask]
    axes[row, 1].plot(peak_freqs, peak_mags, 'rv', markersize=8)
    for f, m in zip(peak_freqs, peak_mags):
        axes[row, 1].annotate(f'{f:.3f}', (f, m), textcoords='offset points',
                              xytext=(5, 5), fontsize=8, color='red')

plt.tight_layout()

## 4. Check harmonic structure
If there are multiple peaks, are they at integer multiples of a fundamental? That would confirm multi-harmonic (comb filter needed).

In [ ]:
# Average magnitude spectrum across many frames for a cleaner view
n_avg = min(100, mov.shape[0])
avg_mag = np.zeros(mov.shape[2] // 2 + 1)

for i in range(n_avg):
    profile = mov[i].astype(np.float32).mean(axis=0)
    profile -= profile.mean()
    mag = np.abs(np.fft.rfft(profile))
    mag[0] = 0
    avg_mag += mag

avg_mag /= n_avg
freqs = np.fft.rfftfreq(mov.shape[2])

# Find peaks using scipy
from scipy.signal import find_peaks
peaks, props = find_peaks(avg_mag, height=avg_mag.max() * 0.1, distance=3)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freqs, avg_mag, 'k-', lw=1)
ax.plot(freqs[peaks], avg_mag[peaks], 'rv', markersize=10)

for p in peaks:
    ax.annotate(f'{freqs[p]:.4f}\n({1/freqs[p]:.1f} px)', (freqs[p], avg_mag[p]),
                textcoords='offset points', xytext=(8, 5), fontsize=9, color='red')

ax.set_xlabel('Frequency (cycles/pixel)')
ax.set_ylabel('Average magnitude')
ax.set_title(f'Average FFT of column profiles ({n_avg} frames)')
ax.set_xlim(0, 0.5)
plt.tight_layout()

# Check harmonic ratios
if len(peaks) >= 2:
    f0 = freqs[peaks[0]]
    print(f"Fundamental: {f0:.4f} cycles/px ({1/f0:.1f} px period)")
    print("Peak ratios relative to fundamental:")
    for p in peaks:
        ratio = freqs[p] / f0
        print(f"  {freqs[p]:.4f}  →  {ratio:.2f}x fundamental  (nearest integer: {round(ratio)})")
else:
    print(f"Single dominant peak at {freqs[peaks[0]]:.4f} cycles/px → likely a clean sinusoid")

## 5. Visualize the drift
Stack column profiles across time as a 2D image — (time × column). If stripes drift left, you'll see diagonal lines.

In [ ]:
# Build (T, W) matrix of de-meaned column profiles
T = mov.shape[0]
W = mov.shape[2]
profiles = np.zeros((T, W), dtype=np.float32)

for i in range(T):
    p = mov[i].astype(np.float32).mean(axis=0)
    p -= p.mean()
    profiles[i] = p

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full time range
ax = axes[0]
vlim = np.percentile(np.abs(profiles), 99)
ax.imshow(profiles, aspect='auto', cmap='RdBu_r', vmin=-vlim, vmax=vlim,
          extent=[0, W, T, 0])
ax.set_xlabel('Column')
ax.set_ylabel('Frame')
ax.set_title('Column profiles over time')

# Zoom in on first ~50 frames to see the drift clearly
n_zoom = min(50, T)
ax = axes[1]
ax.imshow(profiles[:n_zoom], aspect='auto', cmap='RdBu_r', vmin=-vlim, vmax=vlim,
          extent=[0, W, n_zoom, 0])
ax.set_xlabel('Column')
ax.set_ylabel('Frame')
ax.set_title(f'Zoomed (first {n_zoom} frames) — look for diagonal drift')

plt.tight_layout()

## 6. Measure drift rate
Cross-correlate consecutive column profiles to estimate the per-frame pixel shift.

In [ ]:
# Cross-correlation between consecutive frames' column profiles
shifts = []
for i in range(T - 1):
    xcorr = np.correlate(profiles[i], profiles[i + 1], mode='full')
    lags = np.arange(-W + 1, W)
    peak_lag = lags[np.argmax(xcorr)]
    shifts.append(peak_lag)

shifts = np.array(shifts)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(shifts, '.', markersize=2)
axes[0].set_xlabel('Frame pair')
axes[0].set_ylabel('Shift (pixels)')
axes[0].set_title('Per-frame stripe shift (cross-correlation)')
axes[0].axhline(np.median(shifts), color='r', ls='--', label=f'median = {np.median(shifts):.1f} px')
axes[0].legend()

axes[1].hist(shifts, bins=np.arange(shifts.min() - 0.5, shifts.max() + 1.5), edgecolor='k')
axes[1].set_xlabel('Shift (pixels/frame)')
axes[1].set_ylabel('Count')
axes[1].set_title('Shift distribution')

plt.tight_layout()
print(f"Median shift: {np.median(shifts):.2f} px/frame")
print(f"Mean shift:   {np.mean(shifts):.2f} px/frame")

## 7. Per-frame notch destriping — step by step
Apply the Fourier notch approach to a single frame, then compare before/after.

In [ ]:
def destripe_notch(frame, notch_freqs=None, notch_width=0.005):
    """
    Remove vertical banding from a single frame using Fourier notch filtering.
    
    Parameters
    ----------
    frame : (H, W) array
    notch_freqs : list of float, optional
        Frequencies to notch out (cycles/pixel). If None, auto-detect peaks.
    notch_width : float
        Half-width of the notch in frequency space.
    
    Returns
    -------
    cleaned : (H, W) array
    stripe : (H, W) array — what was removed
    info : dict — detected frequencies, magnitudes
    """
    frame = frame.astype(np.float32)
    H, W = frame.shape
    
    # Step 1: column profile (mean of each column across rows)
    col_profile = frame.mean(axis=0)
    col_profile -= col_profile.mean()
    
    # Step 2: FFT of the 1D profile
    spectrum = np.fft.rfft(col_profile)
    freqs = np.fft.rfftfreq(W)
    magnitude = np.abs(spectrum)
    magnitude[0] = 0  # ignore DC
    
    # Step 3: find stripe peaks (auto-detect if not given)
    if notch_freqs is None:
        from scipy.signal import find_peaks
        peak_idx, _ = find_peaks(magnitude, height=magnitude.max() * 0.15, distance=3)
        notch_freqs = freqs[peak_idx].tolist()
    
    # Step 4: build notch filter — zero out frequencies near each peak
    filt = np.ones_like(spectrum)
    for f0 in notch_freqs:
        mask = np.abs(freqs - f0) < notch_width
        filt[mask] = 0.0
    
    # Step 5: apply filter → get cleaned profile
    cleaned_spectrum = spectrum * filt
    cleaned_profile = np.fft.irfft(cleaned_spectrum, n=W)
    
    # The stripe is what we removed from the column profile
    stripe_profile = col_profile - cleaned_profile
    
    # Step 6: subtract stripe from every row
    stripe = np.tile(stripe_profile, (H, 1))
    cleaned = frame - stripe
    
    info = {
        'notch_freqs': notch_freqs,
        'col_profile': col_profile,
        'stripe_profile': stripe_profile,
        'cleaned_profile': cleaned_profile,
        'magnitude': magnitude,
        'freqs': freqs,
    }
    return cleaned, stripe, info

In [ ]:
# Run on a single frame
frame = mov[0].astype(np.float32)
cleaned, stripe, info = destripe_notch(frame)

print(f"Detected stripe frequencies: {info['notch_freqs']}")
for f in info['notch_freqs']:
    print(f"  {f:.4f} cycles/px → period = {1/f:.1f} pixels")

# Show the profiles
fig, axes = plt.subplots(3, 1, figsize=(10, 7))

axes[0].plot(info['col_profile'], label='Original column profile')
axes[0].set_title('Original column profile (de-meaned)')
axes[0].legend()

axes[1].plot(info['stripe_profile'], 'r', label='Stripe component (removed)')
axes[1].set_title('What the notch filter isolated')
axes[1].legend()

axes[2].plot(info['cleaned_profile'], 'g', label='Remaining profile (kept)')
axes[2].set_title('Column profile after notch')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('Column')
plt.tight_layout()

In [ ]:
# Before / after comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

vmin, vmax = np.percentile(frame, [1, 99])
axes[0].imshow(frame, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')

axes[1].imshow(cleaned, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('After notch destripe')

slim = np.percentile(np.abs(stripe), 99)
axes[2].imshow(stripe, cmap='RdBu_r', vmin=-slim, vmax=slim)
axes[2].set_title('Removed stripe pattern')

for ax in axes:
    ax.axis('off')
plt.tight_layout()

## 8. Compare notch vs existing methods on same frame

In [ ]:
from bandaid import destripe_edge, destripe_edge_highpass, destripe_per_column

frame_3d = frame[np.newaxis, :, :]  # existing funcs expect (T,H,W)

results = {
    'Original': frame,
    'edge (mean, n=10)': destripe_edge(frame_3d)[0],
    'edge highpass': destripe_edge_highpass(frame_3d)[0],
    'per-column median': destripe_per_column(frame_3d)[0],
    'notch filter': cleaned,
}

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
for ax, (name, img) in zip(axes, results.items()):
    ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.suptitle('Method comparison — single frame', fontsize=12)
plt.tight_layout()

## 9. What was removed — residual comparison
Look at what each method subtracted. Ideally you see only stripes, no signal.

In [ ]:
fig, axes = plt.subplots(1, len(results) - 1, figsize=(4 * (len(results) - 1), 4))
methods = list(results.items())[1:]  # skip original

for ax, (name, img) in zip(axes, methods):
    residual = frame - img
    rlim = np.percentile(np.abs(residual), 99)
    ax.imshow(residual, cmap='RdBu_r', vmin=-rlim, vmax=rlim)
    ax.set_title(f'{name}\nremoved', fontsize=10)
    ax.axis('off')
plt.suptitle('What each method subtracted (should be stripes only, no cells)', fontsize=12)
plt.tight_layout()